In [1]:
import os

In [2]:
%pwd

'c:\\Users\\p00za\\Desktop\\Collage projects\\Kidney_Disease_Classification_Project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\p00za\\Desktop\\Collage projects\\Kidney_Disease_Classification_Project'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_image_size: list
    params_is_augmentation: bool

In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.Common import read_yaml, create_directories
import tensorflow as tf

In [7]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath=CONFIG_FILE_PATH, 
        params_filepath=PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])

    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(
            self.config.data_ingestion.unzip_dir,"CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone","CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone"
        )

        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir= Path(training.root_dir),
            trained_model_path= Path(training.trained_model_path),
            updated_base_model_path= Path(prepare_base_model.updated_base_model_path),   
            training_data= Path(training_data),
            params_epochs= params.EPOCHS,
            params_is_augmentation= params.AUGMENTATION,    
            params_batch_size= params.BATCH_SIZE,
            params_image_size= params.IMAGE_SIZE,

        )
        return training_config

In [8]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [13]:
from tensorflow.keras.applications.vgg16 import preprocess_input
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )

    
    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            preprocessing_function=preprocess_input,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear",
            class_mode="categorical"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        print("TRAIN PATH:", self.config.training_data)
        print("CLASSES:", os.listdir(self.config.training_data))

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:

            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                preprocessing_function=preprocess_input,
                validation_split=0.20
            )

        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

        print(self.train_generator.class_indices)
    
        
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)



    def train(self):

        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size


        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator,
            
        )

        self.save_model(
            path=self.config.trained_model_path, 
            model=self.model)

In [ ]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()

except Exception as e:
    raise e    

[2026-05-20 17:06:12,710: INFO: Common: yaml file:config\config.yaml loaded successfully]
[2026-05-20 17:06:12,713: INFO: Common: yaml file:params.yaml loaded successfully]
[2026-05-20 17:06:12,715: INFO: Common: created directory at :artifacts]
[2026-05-20 17:06:12,718: INFO: Common: created directory at :artifacts\training]
Model Loaded Successfully

DATASET PATH:
artifacts\data_ingestion\CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone\CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone

FOLDERS INSIDE DATASET:
['Cyst', 'Normal', 'Stone', 'Tumor']
Found 2487 images belonging to 4 classes.
Found 9959 images belonging to 4 classes.

CLASS INDICES:
{'Cyst': 0, 'Normal': 1, 'Stone': 2, 'Tumor': 3}

TRAINING SAMPLES:
9959

VALIDATION SAMPLES:
2487

TRAIN LABELS:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

STARTING TRAINING...

Epoch 1/16
622/622 [==============================] - 1744s 3s/step - loss: 2.3596 - accuracy: 0.3839 - val_loss: 2.6366 - val_accuracy: 0.3129
Epoch 2/16
622/622 [=================

In [15]:
import os
import tensorflow as tf

from pathlib import Path
from dataclasses import dataclass
from tensorflow.keras.applications.vgg16 import preprocess_input
from cnnClassifier.entity.config_entity import TrainingConfig


class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):

        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )

        print("Model Loaded Successfully")


    def train_valid_generator(self):

        # =========================
        # CHECK DATASET PATH
        # =========================

        print("\nDATASET PATH:")
        print(self.config.training_data)

        print("\nFOLDERS INSIDE DATASET:")
        print(os.listdir(self.config.training_data))

        # Expected:
        # ['Cyst', 'Normal', 'Stone', 'Tumor']


        # =========================
        # IMAGE GENERATOR SETTINGS
        # =========================

        datagenerator_kwargs = dict(
            preprocessing_function=preprocess_input,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear",
            class_mode="categorical",
            seed=42
        )


        # =========================
        # VALIDATION GENERATOR
        # =========================

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )


        # =========================
        # TRAIN GENERATOR
        # =========================

        if self.config.params_is_augmentation:

            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                preprocessing_function=preprocess_input,
                validation_split=0.20,
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2
            )

        else:

            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                preprocessing_function=preprocess_input,
                validation_split=0.20
            )


        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )


        # =========================
        # DEBUGGING OUTPUT
        # =========================

        print("\nCLASS INDICES:")
        print(self.train_generator.class_indices)

        print("\nTRAINING SAMPLES:")
        print(self.train_generator.samples)

        print("\nVALIDATION SAMPLES:")
        print(self.valid_generator.samples)

        print("\nTRAIN LABELS:")
        print(self.train_generator.labels[:20])


    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):

        model.save(path)

        print(f"\nModel Saved at: {path}")


    def train(self):

        self.steps_per_epoch = (
            self.train_generator.samples //
            self.train_generator.batch_size
        )

        self.validation_steps = (
            self.valid_generator.samples //
            self.valid_generator.batch_size
        )

        print("\nSTARTING TRAINING...\n")

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_data=self.valid_generator,
            validation_steps=self.validation_steps
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )